<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 48px 40px; border-radius: 16px; color: white; text-align: center; margin-bottom: 32px;">
  <h1 style="font-size: 2.4em; font-weight: 800; letter-spacing: -0.5px; margin: 0 0 12px 0;">🛸 Multi-Modal Drone-Based Cell Tower Inspection Pipeline</h1>
  <p style="font-size: 1.2em; color: #a8c8f0; margin: 0 0 8px 0;">Simultaneous RGB • Thermal • Geospatial Telemetry Fusion</p>
  <p style="font-size: 0.95em; color: #7a9cbf; margin: 0;">Real-Time Structural Anomaly Detection &amp; Classification using YOLOv8</p>
</div>

---

## Abstract

This notebook presents a complete end-to-end pipeline for automated structural inspection of 5G/LTE cell towers using multi-rotor drones. The system ingests three concurrent data streams — high-resolution 4K RGB imagery, thermal infrared video, and geospatial telemetry — and fuses them using a synchronized frame-ID scheme to produce a unified anomaly report per inspection waypoint.

Two YOLOv8 detection heads are trained independently: one on RGB imagery for visual structural defects (cracks, rust, corrosion, missing bolts, paint damage, structural bends) and one on thermal false-colour images for heat-signature anomalies (hotspots, connector failures, equipment overheating). Telemetry data provides the spatial context layer — GPS coordinates, altitude, gimbal angles, and environmental readings are fused at inference time to produce georeferenced, severity-ranked inspection reports.

The pipeline is designed so that training happens offline on the three labelled datasets, while inference runs on any image or short video clip captured during an inspection flight.

---

## Table of Contents

| # | Section |
|---|---|
| 1 | System Architecture & Data Flow |
| 2 | Folder Structure & Dataset Overview |
| 3 | Environment Setup & Dependencies |
| 4 | Dataset Exploration & Visualisation |
| 5 | RGB Model — Training (YOLOv8) |
| 6 | Thermal Model — Training (YOLOv8) |
| 7 | Telemetry Parser & Feature Engineering |
| 8 | Multi-Modal Fusion Logic |
| 9 | Inference on Static Images |
| 10 | Severity Scoring & Risk Matrix |
| 11 | Georeferenced Inspection Report |
| 12 | Key Formulas & Theoretical Grounding |

---

## 1. System Architecture & Data Flow

```
┌─────────────────────────────────────────────────────────────────────────────────┐
│                   DRONE INSPECTION — MULTI-MODAL DATA PIPELINE                  │
├─────────────────┬───────────────────────┬───────────────────────────────────────┤
│  STREAM A       │  STREAM B             │  STREAM C                             │
│  4K RGB Camera  │  Thermal (FLIR)       │  Geospatial Telemetry                │
│  640×640 PNG    │  640×512 JPG          │  63 fields @ 10 Hz                   │
│  6 defect cls   │  8 thermal cls        │  GPS / IMU / Battery / Motors        │
│       │         │        │              │            │                          │
│       ▼         │        ▼              │            ▼                          │
│  YOLOv8-RGB     │  YOLOv8-Thermal       │  Telemetry Parser                    │
│  Detector       │  Detector             │  + Feature Engineering               │
│       │         │        │              │            │                          │
└───────┼─────────┴────────┼──────────────┴────────────┼──────────────────────────┘
        │                  │                            │
        └──────────────────┴────────────────────────────┘
                                    │
                       ┌────────────▼────────────┐
                       │   FUSION ENGINE          │
                       │  sync: mission_id +      │
                       │        frame_id           │
                       └────────────┬────────────┘
                                    │
                       ┌────────────▼────────────┐
                       │  Severity Scoring &      │
                       │  Risk Matrix             │
                       └────────────┬────────────┘
                                    │
                       ┌────────────▼────────────┐
                       │  Georeferenced Report    │
                       │  (JSON + Map + CSV)      │
                       └─────────────────────────┘
```

### Design Principles

**Modality Independence** — each stream is processed by its own model trained exclusively on that modality's annotations. This prevents cross-modal confusion during gradient updates.

**Synchronisation Key** — `mission_id` + `frame_id` is the join key across all three streams. Every telemetry record, RGB frame, and thermal frame share this composite key, allowing exact temporal alignment.

**Late Fusion** — detection results from both YOLO models are merged *after* inference rather than during feature extraction. This preserves each model's optimised representations and makes the system fault-tolerant (one stream can fail without breaking the pipeline).

**Severity Escalation** — fused detections are ranked by a composite score combining visual confidence, thermal temperature delta, and telemetry-derived proximity to the anomaly.

---

## 2. Folder Structure & Dataset Overview

The pipeline expects the following directory layout. All paths below are relative to the project root.

```
project_root/
│
├── tower_inspection_dataset/          ← RGB visual defect dataset
│   ├── dataset.yaml
│   ├── train/
│   │   ├── images/  (640×640 PNG)
│   │   └── labels/  (YOLO .txt)
│   ├── val/
│   └── test/
│
├── cell_tower_thermal/                ← Thermal infrared dataset
│   ├── dataset.yaml
│   ├── train/
│   │   ├── images/  (640×512 JPG, iron/rainbow palette)
│   │   └── labels/  (YOLO .txt)
│   ├── val/
│   └── test/
│
├── telemetry_dataset/                 ← Geospatial telemetry
│   ├── dataset.yaml
│   ├── schema.json
│   ├── mission_index.csv
│   ├── train/
│   │   ├── telemetry/        (70 × CSV, 63 fields @ 10 Hz)
│   │   ├── annotations/      (70 × JSON anomaly files)
│   │   └── mission_metadata/ (70 × JSON)
│   ├── val/
│   └── test/
│
├── runs/                              ← YOLOv8 training outputs (auto-created)
│   ├── rgb_tower/
│   └── thermal_tower/
│
└── drone_tower_inspection.ipynb       ← This notebook
```

### Dataset Quick Reference

| Property | RGB Dataset | Thermal Dataset | Telemetry Dataset |
|---|---|---|---|
| Total samples | ~640 images | 100 images | 100 missions |
| Resolution | 640 × 640 px | 640 × 512 px | 63 fields/frame @ 10 Hz |
| Train split | 70 % | 70 (images) | 70 missions |
| Val split | 15 % | 20 (images) | 20 missions |
| Test split | 15 % | 10 (images) | 10 missions |
| Annotation | YOLO xywh | YOLO xywh | JSON + CSV |
| Classes | 6 | 8 | 9 |
| Label format | Normalised [0,1] | Normalised [0,1] | Structured CSV/JSON |

### Class Mappings

**RGB Classes (visual defects)**

| ID | Class | Description | Severity |
|---|---|---|---|
| 0 | `crack` | Fractures on members / welds | 🔴 URGENT |
| 1 | `rust` | Surface iron-oxide discoloration | 🟡 Monitor |
| 2 | `corrosion` | Advanced electrochemical degradation | 🟠 High |
| 3 | `missing_bolt` | Empty fastener holes | 🔴 URGENT |
| 4 | `paint_damage` | Chipped / peeling protective coating | 🟢 Low |
| 5 | `structural_bend` | Buckled or kinked structural members | 🔴 URGENT |

**Thermal Classes**

| ID | Class | Description | Severity |
|---|---|---|---|
| 0 | `tower_structure` | Lattice / monopole body | — Normal |
| 1 | `antenna_panel` | Sector antennas, panel arrays | — Normal |
| 2 | `cable_harness` | Feeder cables, coax bundles | — Normal |
| 3 | `hotspot_critical` | >15 °C above ambient | 🔴 URGENT |
| 4 | `hotspot_moderate` | 8–15 °C above ambient | 🟠 High |
| 5 | `hotspot_minor` | <8 °C above ambient | 🟡 Monitor |
| 6 | `connector_joint` | Weatherproof connectors | — Normal |
| 7 | `equipment_cabinet` | BTS / RRU / power cabinets | — Normal |

**Telemetry Anomaly Classes**

| ID | Class | Severity |
|---|---|---|
| 0 | `hotspot_critical` | 🔴 URGENT |
| 1 | `hotspot_moderate` | 🟠 High |
| 2 | `hotspot_minor` | 🟡 Medium |
| 3 | `antenna_misalignment` | 🟠 High |
| 4 | `cable_fault` | 🔴 URGENT |
| 5 | `connector_failure` | 🔴 URGENT |
| 6 | `corrosion` | 🟡 Medium |
| 7 | `structural_crack` | 🔴 URGENT |
| 8 | `no_anomaly` | ✅ Normal |

---

## 3. Environment Setup & Dependencies

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3.1 — Install all required packages
# Run once; restart kernel if prompted
# ─────────────────────────────────────────────────────────────
import subprocess, sys

packages = [
    "ultralytics",     # YOLOv8
    "opencv-python",   # image / video I/O
    "pandas",          # telemetry CSV parsing
    "numpy",           # numerical ops
    "matplotlib",      # plotting
    "seaborn",         # statistical charts
    "folium",          # georeferenced map rendering
    "Pillow",          # image utilities
    "scipy",           # signal processing for telemetry
    "scikit-learn",    # evaluation metrics
    "pyyaml",          # dataset.yaml parsing
    "tqdm",            # progress bars
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages installed successfully.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3.2 — Imports & global configuration
# ─────────────────────────────────────────────────────────────
import os, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import yaml
import folium
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from scipy import stats
from sklearn.metrics import confusion_matrix, classification_report
from ultralytics import YOLO
from IPython.display import display, HTML, Image as IPImage

warnings.filterwarnings("ignore")

# ── Plotting style ──────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
PALETTE = ["#e63946", "#f4a261", "#2a9d8f", "#264653", "#e9c46a", "#a8dadc", "#457b9d", "#1d3557"]

# ── Dataset root paths ──────────────────────────────────────
ROOT          = Path(".")                              # project root
RGB_YAML      = ROOT / "tower_inspection_dataset" / "dataset.yaml"
THERMAL_YAML  = ROOT / "cell_tower_thermal"       / "dataset.yaml"
TELEMETRY_DIR = ROOT / "telemetry_dataset"

# ── Class metadata ──────────────────────────────────────────
RGB_CLASSES      = ["crack", "rust", "corrosion", "missing_bolt", "paint_damage", "structural_bend"]
THERMAL_CLASSES  = ["tower_structure", "antenna_panel", "cable_harness",
                    "hotspot_critical", "hotspot_moderate", "hotspot_minor",
                    "connector_joint", "equipment_cabinet"]
TELEMETRY_CLASSES = ["hotspot_critical", "hotspot_moderate", "hotspot_minor",
                     "antenna_misalignment", "cable_fault", "connector_failure",
                     "corrosion", "structural_crack", "no_anomaly"]

SEVERITY_MAP = {
    "crack": 4, "missing_bolt": 4, "structural_bend": 4,
    "corrosion": 3, "rust": 2, "paint_damage": 1,
    "hotspot_critical": 4, "cable_fault": 4, "connector_failure": 4, "structural_crack": 4,
    "hotspot_moderate": 3, "antenna_misalignment": 3,
    "hotspot_minor": 2, "no_anomaly": 0,
}

SEVERITY_LABEL = {4: "🔴 URGENT", 3: "🟠 High", 2: "🟡 Medium", 1: "🟢 Low", 0: "✅ Normal"}

print("✅ Configuration complete.")
print(f"   RGB classes      : {RGB_CLASSES}")
print(f"   Thermal classes  : {THERMAL_CLASSES}")
print(f"   Telemetry classes: {TELEMETRY_CLASSES}")

---

## 4. Dataset Exploration & Visualisation

Before training any model, understanding the data distribution prevents common pitfalls such as training on severely imbalanced classes without appropriate augmentation or loss weighting.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.1 — Count annotations per class (RGB dataset)
# ─────────────────────────────────────────────────────────────
def count_labels(label_dir, n_classes):
    """Scan all .txt YOLO label files and tally class frequencies."""
    counts = np.zeros(n_classes, dtype=int)
    label_files = list(Path(label_dir).rglob("*.txt"))
    for lf in label_files:
        lines = lf.read_text().strip().split("\n")
        for line in lines:
            if line.strip():
                cls_id = int(line.split()[0])
                if cls_id < n_classes:
                    counts[cls_id] += 1
    return counts, label_files

# RGB
rgb_train_counts, rgb_train_files = count_labels(
    ROOT / "tower_inspection_dataset" / "train" / "labels", len(RGB_CLASSES))
rgb_val_counts, _  = count_labels(
    ROOT / "tower_inspection_dataset" / "val" / "labels",   len(RGB_CLASSES))

# Thermal
th_train_counts, th_train_files = count_labels(
    ROOT / "cell_tower_thermal" / "train" / "labels", len(THERMAL_CLASSES))
th_val_counts, _   = count_labels(
    ROOT / "cell_tower_thermal" / "val"   / "labels", len(THERMAL_CLASSES))

print("RGB Train label distribution:")
for cls, cnt in zip(RGB_CLASSES, rgb_train_counts):
    print(f"  {cls:<20} → {cnt:>4} instances")
print(f"\n  Total RGB train labels : {rgb_train_counts.sum()}")
print(f"  Total RGB train images : {len(rgb_train_files)}")

print("\nThermal Train label distribution:")
for cls, cnt in zip(THERMAL_CLASSES, th_train_counts):
    print(f"  {cls:<22} → {cnt:>4} instances")
print(f"\n  Total Thermal train labels : {th_train_counts.sum()}")
print(f"  Total Thermal train images : {len(th_train_files)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.2 — Class distribution bar charts
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Annotation Class Distribution — Train Split", fontsize=14, fontweight="bold", y=1.02)

# RGB
bars_rgb = axes[0].bar(RGB_CLASSES, rgb_train_counts, color=PALETTE[:6], edgecolor="white", linewidth=0.8)
axes[0].set_title("RGB / Visual Defect Dataset", fontweight="bold")
axes[0].set_ylabel("Instance count")
axes[0].set_xlabel("Defect class")
axes[0].tick_params(axis='x', rotation=30)
for bar, v in zip(bars_rgb, rgb_train_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(v),
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Thermal
bars_th = axes[1].bar(THERMAL_CLASSES, th_train_counts, color=PALETTE, edgecolor="white", linewidth=0.8)
axes[1].set_title("Thermal Infrared Dataset", fontweight="bold")
axes[1].set_ylabel("Instance count")
axes[1].set_xlabel("Thermal class")
axes[1].tick_params(axis='x', rotation=35)
for bar, v in zip(bars_th, th_train_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(v),
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig("class_distribution.png", bbox_inches="tight")
plt.show()
print("Chart saved → class_distribution.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.3 — Display sample RGB images with bounding boxes
# ─────────────────────────────────────────────────────────────
def draw_yolo_boxes(img_path, label_path, class_names, palette=None):
    """Render YOLO normalised boxes onto an image and return it."""
    if palette is None:
        palette = [(230,57,70), (244,162,97), (42,157,143), (38,70,83), (233,196,106), (168,218,220)]
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    if Path(label_path).exists():
        for line in Path(label_path).read_text().strip().split("\n"):
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
            color = palette[cls_id % len(palette)]
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            cv2.putText(img, label, (x1, max(y1-6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Collect up to 6 random RGB images
rgb_img_dir   = ROOT / "tower_inspection_dataset" / "train" / "images"
rgb_lbl_dir   = ROOT / "tower_inspection_dataset" / "train" / "labels"
rgb_imgs      = sorted(rgb_img_dir.glob("*.png"))[:6] if rgb_img_dir.exists() else []

if rgb_imgs:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("RGB Dataset — Ground-Truth Annotations", fontsize=13, fontweight="bold")
    for ax, img_path in zip(axes.flat, rgb_imgs):
        lbl_path = rgb_lbl_dir / (img_path.stem + ".txt")
        rendered = draw_yolo_boxes(img_path, lbl_path, RGB_CLASSES)
        if rendered is not None:
            ax.imshow(rendered)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("rgb_samples.png", bbox_inches="tight")
    plt.show()
    print("Saved → rgb_samples.png")
else:
    print("ℹ️  No RGB images found — run after placing the dataset.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.4 — Display sample Thermal images with bounding boxes
# ─────────────────────────────────────────────────────────────
th_img_dir = ROOT / "cell_tower_thermal" / "train" / "images"
th_lbl_dir = ROOT / "cell_tower_thermal" / "train" / "labels"
th_imgs    = sorted(th_img_dir.glob("*.jpg"))[:6] if th_img_dir.exists() else []

# Thermal palette — warm→hot colour scale suggestion
th_palette = [(255,255,0),(255,165,0),(255,69,0),(200,0,0),(128,0,128),(0,191,255),(0,128,0),(128,128,128)]

if th_imgs:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle("Thermal Dataset — Ground-Truth Annotations", fontsize=13, fontweight="bold")
    for ax, img_path in zip(axes.flat, th_imgs):
        lbl_path = th_lbl_dir / (img_path.stem + ".txt")
        rendered = draw_yolo_boxes(img_path, lbl_path, THERMAL_CLASSES, palette=th_palette)
        if rendered is not None:
            ax.imshow(rendered)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig("thermal_samples.png", bbox_inches="tight")
    plt.show()
    print("Saved → thermal_samples.png")
else:
    print("ℹ️  No thermal images found — run after placing the dataset.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.5 — Telemetry exploration
# ─────────────────────────────────────────────────────────────
def load_telemetry_split(split="train"):
    """Load all mission CSVs for a given split and concatenate."""
    tel_dir = TELEMETRY_DIR / split / "telemetry"
    csvs    = list(tel_dir.glob("*.csv")) if tel_dir.exists() else []
    if not csvs:
        return pd.DataFrame()
    dfs = []
    for c in csvs:
        df = pd.read_csv(c)
        df["source_file"] = c.name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

tel_train = load_telemetry_split("train")

if not tel_train.empty:
    print(f"Telemetry train rows : {len(tel_train):,}")
    print(f"Telemetry columns    : {list(tel_train.columns)[:10]} ...")
    display(tel_train[["timestamp_utc", "mission_id", "frame_id", "lat", "lon",
                        "altitude_m", "heading", "battery_remaining_pct"]].head(8))
else:
    print("ℹ️  Telemetry CSVs not found — place dataset files and re-run.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4.6 — Telemetry signal visualisation (altitude & battery)
# ─────────────────────────────────────────────────────────────
if not tel_train.empty and "altitude_m" in tel_train.columns:
    # pick one mission for clarity
    mission_ids = tel_train["mission_id"].unique()
    sample_mission = tel_train[tel_train["mission_id"] == mission_ids[0]].copy()
    sample_mission = sample_mission.reset_index(drop=True)

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(f"Telemetry Signal — Mission: {mission_ids[0]}", fontweight="bold")

    axes[0].plot(sample_mission.index, sample_mission["altitude_m"], color="#457b9d", lw=1.5)
    axes[0].set_ylabel("Altitude (m AGL)")
    axes[0].fill_between(sample_mission.index, sample_mission["altitude_m"], alpha=0.15, color="#457b9d")

    if "battery_remaining_pct" in sample_mission.columns:
        axes[1].plot(sample_mission.index, sample_mission["battery_remaining_pct"],
                     color="#2a9d8f", lw=1.5)
        axes[1].set_ylabel("Battery (%)")
        axes[1].fill_between(sample_mission.index, sample_mission["battery_remaining_pct"],
                              alpha=0.15, color="#2a9d8f")

    if "ground_speed" in sample_mission.columns:
        axes[2].plot(sample_mission.index, sample_mission["ground_speed"],
                     color="#e63946", lw=1.5)
        axes[2].set_ylabel("Ground Speed (m/s)")
        axes[2].set_xlabel("Frame index (10 Hz)")

    plt.tight_layout()
    plt.savefig("telemetry_signals.png", bbox_inches="tight")
    plt.show()
    print("Saved → telemetry_signals.png")
else:
    print("ℹ️  Skipping telemetry plot — data not available.")

---

## 5. RGB Model — Training (YOLOv8)

### 5.1 Why YOLOv8?

YOLOv8 (You Only Look Once, version 8) uses a **CSPDarknet53 backbone** with a **C2f neck** and an **anchor-free decoupled detection head**. The anchor-free design is well-suited here because structural defects on towers appear at highly variable aspect ratios and scales — a single crack may span 5 % of the image width while a rusted section spans 40 %.

### 5.2 Key Training Formulas

**IoU (Intersection over Union)**

$$\text{IoU}(B_{pred}, B_{gt}) = \frac{|B_{pred} \cap B_{gt}|}{|B_{pred} \cup B_{gt}|}$$

Used to match predicted boxes to ground-truth boxes during loss computation. A prediction is considered a True Positive when IoU ≥ 0.5 (default threshold).

**CIoU Loss (Complete IoU)**

$$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(b, b^{gt})}{c^2} + \alpha v$$

where:
- $\rho^2(b, b^{gt})$ = squared Euclidean distance between box centres
- $c$ = diagonal of the smallest enclosing box
- $v = \frac{4}{\pi^2}\left(\arctan\frac{w^{gt}}{h^{gt}} - \arctan\frac{w}{h}\right)^2$ penalises aspect-ratio divergence
- $\alpha = \frac{v}{1 - \text{IoU} + v}$ is the trade-off parameter

**Binary Cross-Entropy Loss (classification)**

$$\mathcal{L}_{\text{cls}} = -\sum_{c} \left[ y_c \log(\hat{p}_c) + (1-y_c) \log(1-\hat{p}_c) \right]$$

**Focal Loss modifier (for class imbalance)**

$$\mathcal{L}_{\text{focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

Here $\gamma = 2.0$ down-weights easy negatives (background regions) so the model focuses on hard positive defect instances. This is critical because for every crack pixel there are roughly 50× more background pixels.

**Total YOLOv8 Loss**

$$\mathcal{L}_{\text{total}} = \lambda_{\text{box}} \cdot \mathcal{L}_{\text{CIoU}} + \lambda_{\text{cls}} \cdot \mathcal{L}_{\text{cls}} + \lambda_{\text{dfl}} \cdot \mathcal{L}_{\text{DFL}}$$

where $\mathcal{L}_{\text{DFL}}$ is the Distribution Focal Loss on the four bounding-box edge distributions. Default weights: $\lambda_{\text{box}} = 7.5$, $\lambda_{\text{cls}} = 0.5$, $\lambda_{\text{dfl}} = 1.5$.

**Mean Average Precision**

$$\text{mAP}_{50} = \frac{1}{N_c} \sum_{c=1}^{N_c} \text{AP}_{c} \quad \text{where} \quad \text{AP}_c = \int_0^1 P_c(r)\, dr$$

$P_c(r)$ is the precision-recall curve for class $c$ at IoU threshold 0.50. mAP@50:95 averages across IoU thresholds [0.50, 0.55, …, 0.95].

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5.1 — Verify RGB dataset.yaml exists; create if missing
# ─────────────────────────────────────────────────────────────
rgb_yaml_content = {
    "path":  str(ROOT / "tower_inspection_dataset"),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    6,
    "names": RGB_CLASSES,
}

if not RGB_YAML.exists():
    RGB_YAML.parent.mkdir(parents=True, exist_ok=True)
    with open(RGB_YAML, "w") as f:
        yaml.dump(rgb_yaml_content, f, default_flow_style=False)
    print(f"Created: {RGB_YAML}")
else:
    print(f"Found  : {RGB_YAML}")

with open(RGB_YAML) as f:
    print("\ndataset.yaml contents:")
    print(yaml.dump(yaml.safe_load(f), default_flow_style=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5.2 — Train YOLOv8n on RGB dataset
# Training takes ~20-40 min on GPU (RTX 3060 / T4 class)
# ─────────────────────────────────────────────────────────────
rgb_model = YOLO("yolov8n.pt")   # nano backbone — swap to yolov8s.pt for higher accuracy

rgb_results = rgb_model.train(
    data      = str(RGB_YAML),
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    lr0       = 1e-3,        # initial learning rate
    lrf       = 0.01,        # final lr = lr0 * lrf (cosine annealing)
    warmup_epochs = 3,       # linear warmup to avoid instability
    mosaic    = 1.0,         # mosaic augmentation probability
    flipud    = 0.0,         # no vertical flip (towers are vertical structures)
    fliplr    = 0.5,         # horizontal flip
    degrees   = 5.0,         # mild rotation ±5°
    scale     = 0.5,         # scale jitter 50 %
    hsv_h     = 0.015,       # hue jitter
    hsv_s     = 0.7,         # saturation jitter
    hsv_v     = 0.4,         # brightness jitter
    patience  = 20,          # early stop if no improvement for 20 epochs
    project   = "runs",
    name      = "rgb_tower",
    exist_ok  = True,
    verbose   = True,
)

print("\n✅ RGB model training complete.")
print(f"Best weights saved at: runs/rgb_tower/weights/best.pt")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5.3 — Plot RGB training curves
# ─────────────────────────────────────────────────────────────
results_csv = Path("runs/rgb_tower/results.csv")

if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    fig.suptitle("RGB YOLOv8 — Training Metrics", fontsize=14, fontweight="bold")

    metrics = [
        ("train/box_loss",   "Box Loss (train)",   "#e63946"),
        ("train/cls_loss",   "Class Loss (train)",  "#f4a261"),
        ("train/dfl_loss",   "DFL Loss (train)",    "#2a9d8f"),
        ("metrics/mAP50(B)", "mAP@0.50",            "#457b9d"),
        ("metrics/mAP50-95(B)", "mAP@0.50:0.95",   "#264653"),
        ("metrics/precision(B)","Precision",         "#a8dadc"),
    ]

    for ax, (col, title, color) in zip(axes.flat, metrics):
        if col in df_res.columns:
            ax.plot(df_res["epoch"], df_res[col], color=color, lw=2)
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Epoch")
        else:
            ax.text(0.5, 0.5, f"{col}\nnot found", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)

    plt.tight_layout()
    plt.savefig("rgb_training_curves.png", bbox_inches="tight")
    plt.show()
    print("Saved → rgb_training_curves.png")
else:
    print("ℹ️  Training results not found — run Cell 5.2 first.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5.4 — Validate RGB model & show per-class mAP table
# ─────────────────────────────────────────────────────────────
rgb_best = Path("runs/rgb_tower/weights/best.pt")

if rgb_best.exists():
    rgb_eval_model = YOLO(str(rgb_best))
    val_metrics    = rgb_eval_model.val(data=str(RGB_YAML), imgsz=640, verbose=False)

    per_class_ap50 = val_metrics.box.ap50  # AP50 per class array
    per_class_ap   = val_metrics.box.ap    # AP50:95 per class array

    df_val = pd.DataFrame({
        "Class"     : RGB_CLASSES[:len(per_class_ap50)],
        "AP@0.50"   : [round(x, 4) for x in per_class_ap50],
        "AP@50:95"  : [round(x, 4) for x in per_class_ap],
    })
    df_val.loc[len(df_val)] = ["**mAP (mean)**",
                                round(float(val_metrics.box.map50), 4),
                                round(float(val_metrics.box.map), 4)]

    print("\nRGB Model — Validation Results")
    print("=" * 42)
    print(f"  mAP@0.50     : {val_metrics.box.map50:.4f}")
    print(f"  mAP@50:95    : {val_metrics.box.map:.4f}")
    print(f"  Precision    : {val_metrics.box.mp:.4f}")
    print(f"  Recall       : {val_metrics.box.mr:.4f}")
    print("\nPer-class breakdown:")
    display(df_val.style.background_gradient(subset=["AP@0.50", "AP@50:95"], cmap="YlGn"))
else:
    print("ℹ️  Model not found — run Cell 5.2 first.")

---

## 6. Thermal Model — Training (YOLOv8)

### Thermal-Specific Considerations

Thermal images (false-colour iron/rainbow palette) have fundamentally different statistical properties from RGB images:

| Property | RGB | Thermal |
|---|---|---|
| Pixel encoding | Visible light reflectance | Radiated heat intensity |
| Resolution | 640 × 640 | 640 × 512 |
| Colour meaning | Physical colour | Temperature proxy |
| Texture cues | Rich (rust, cracks visible) | Smooth (temperature gradients) |
| Class focus | 6 structural defects | 8 classes (3 anomaly + 5 structural context) |

Because thermal images have lower texture variance, we reduce mosaic augmentation probability and increase the contrast/brightness jitter range to simulate different atmospheric conditions and emissivity variations.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6.1 — Verify Thermal dataset.yaml
# ─────────────────────────────────────────────────────────────
th_yaml_content = {
    "path":  str(ROOT / "cell_tower_thermal"),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    8,
    "names": THERMAL_CLASSES,
}

if not THERMAL_YAML.exists():
    THERMAL_YAML.parent.mkdir(parents=True, exist_ok=True)
    with open(THERMAL_YAML, "w") as f:
        yaml.dump(th_yaml_content, f, default_flow_style=False)
    print(f"Created: {THERMAL_YAML}")
else:
    print(f"Found  : {THERMAL_YAML}")

with open(THERMAL_YAML) as f:
    print("\ndataset.yaml contents:")
    print(yaml.dump(yaml.safe_load(f), default_flow_style=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6.2 — Train YOLOv8n on Thermal dataset
# ─────────────────────────────────────────────────────────────
thermal_model = YOLO("yolov8n.pt")

thermal_results = thermal_model.train(
    data         = str(THERMAL_YAML),
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,
    lr0          = 1e-3,
    lrf          = 0.01,
    warmup_epochs= 3,
    mosaic       = 0.5,     # reduced for thermal (less useful with heat maps)
    flipud       = 0.0,
    fliplr       = 0.5,
    degrees      = 3.0,     # minimal rotation (thermal gradients are directional)
    scale        = 0.4,
    hsv_h        = 0.0,     # hue is meaningful in false-colour thermal — don't jitter
    hsv_s        = 0.3,
    hsv_v        = 0.5,     # brightness jitter simulates emissivity changes
    patience     = 20,
    project      = "runs",
    name         = "thermal_tower",
    exist_ok     = True,
    verbose      = True,
)

print("\n✅ Thermal model training complete.")
print(f"Best weights saved at: runs/thermal_tower/weights/best.pt")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6.3 — Plot Thermal training curves
# ─────────────────────────────────────────────────────────────
th_results_csv = Path("runs/thermal_tower/results.csv")

if th_results_csv.exists():
    df_th = pd.read_csv(th_results_csv)
    df_th.columns = df_th.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    fig.suptitle("Thermal YOLOv8 — Training Metrics", fontsize=14, fontweight="bold")

    metrics = [
        ("train/box_loss",      "Box Loss (train)",    "#e63946"),
        ("train/cls_loss",      "Class Loss (train)",   "#f4a261"),
        ("train/dfl_loss",      "DFL Loss (train)",     "#2a9d8f"),
        ("metrics/mAP50(B)",    "mAP@0.50",             "#457b9d"),
        ("metrics/mAP50-95(B)", "mAP@0.50:0.95",        "#264653"),
        ("val/cls_loss",        "Class Loss (val)",      "#a8dadc"),
    ]

    for ax, (col, title, color) in zip(axes.flat, metrics):
        if col in df_th.columns:
            ax.plot(df_th["epoch"], df_th[col], color=color, lw=2)
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Epoch")
        else:
            ax.text(0.5, 0.5, f"{col}\nnot found", ha="center", va="center",
                    transform=ax.transAxes, color="gray")
            ax.set_title(title)

    plt.tight_layout()
    plt.savefig("thermal_training_curves.png", bbox_inches="tight")
    plt.show()
    print("Saved → thermal_training_curves.png")
else:
    print("ℹ️  Thermal training results not found — run Cell 6.2 first.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6.4 — Validate Thermal model
# ─────────────────────────────────────────────────────────────
th_best = Path("runs/thermal_tower/weights/best.pt")

if th_best.exists():
    th_eval_model  = YOLO(str(th_best))
    th_val_metrics = th_eval_model.val(data=str(THERMAL_YAML), imgsz=640, verbose=False)

    th_ap50 = th_val_metrics.box.ap50
    th_ap   = th_val_metrics.box.ap

    df_th_val = pd.DataFrame({
        "Class"   : THERMAL_CLASSES[:len(th_ap50)],
        "AP@0.50" : [round(x, 4) for x in th_ap50],
        "AP@50:95": [round(x, 4) for x in th_ap],
    })
    df_th_val.loc[len(df_th_val)] = ["**mAP (mean)**",
                                      round(float(th_val_metrics.box.map50), 4),
                                      round(float(th_val_metrics.box.map), 4)]

    print("\nThermal Model — Validation Results")
    print("=" * 42)
    print(f"  mAP@0.50     : {th_val_metrics.box.map50:.4f}")
    print(f"  mAP@50:95    : {th_val_metrics.box.map:.4f}")
    print(f"  Precision    : {th_val_metrics.box.mp:.4f}")
    print(f"  Recall       : {th_val_metrics.box.mr:.4f}")
    print("\nPer-class breakdown:")
    display(df_th_val.style.background_gradient(subset=["AP@0.50", "AP@50:95"], cmap="YlOrRd"))
else:
    print("ℹ️  Thermal model not found — run Cell 6.2 first.")

---

## 7. Telemetry Parser & Feature Engineering

Raw telemetry CSVs contain 63 fields per frame sampled at 10 Hz. The parser extracts:

1. **Spatial context** — GPS (lat, lon, altitude), heading, distance to tower
2. **Gimbal state** — pitch and roll determine the camera viewing angle at the time of each frame
3. **Environmental context** — wind speed, temperature, humidity (affect detection confidence)
4. **Pre-labelled anomaly fields** — `detected`, `class`, `confidence`, `bbox_xywh`, `gps_tag`, `ir_temp_delta`

### Camera Projection Formula

To map a 2D detected bounding box back to 3D world coordinates, we apply a simplified pinhole camera + gimbal model:

$$\mathbf{P}_{world} = \mathbf{R}_{drone} \cdot \mathbf{R}_{gimbal} \cdot \mathbf{K}^{-1} \cdot \tilde{\mathbf{p}}_{px} \cdot d$$

where:
- $\mathbf{K}$ is the camera intrinsic matrix (focal length, principal point)
- $\tilde{\mathbf{p}}_{px}$ is the homogeneous pixel coordinate of the box centre
- $\mathbf{R}_{drone}$ is the drone attitude rotation matrix built from (roll, pitch, yaw)
- $\mathbf{R}_{gimbal}$ compensates for gimbal stabilisation offsets
- $d$ is the slant range distance to the tower surface

In practice, for inspection reporting we use the drone's GPS coordinate tagged at the frame timestamp as the anomaly's geolocation (precision ≈ 0.5–2 m).

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7.1 — Telemetry parser class
# ─────────────────────────────────────────────────────────────
class TelemetryParser:
    """Loads a single mission CSV and exposes frame-level telemetry."""

    REQUIRED_COLS = ["mission_id", "frame_id", "lat", "lon", "altitude_m"]

    def __init__(self, csv_path):
        self.path = Path(csv_path)
        self.df   = pd.read_csv(csv_path) if self.path.exists() else pd.DataFrame()
        self._validate()

    def _validate(self):
        if self.df.empty:
            return
        missing = [c for c in self.REQUIRED_COLS if c not in self.df.columns]
        if missing:
            print(f"⚠️  Telemetry missing columns: {missing}")

    def get_frame(self, frame_id):
        """Return a dict of telemetry fields for a specific frame."""
        row = self.df[self.df["frame_id"] == frame_id]
        if row.empty:
            return None
        return row.iloc[0].to_dict()

    def get_context(self, frame_id):
        """Return just the spatial + environmental context."""
        frame = self.get_frame(frame_id)
        if frame is None:
            return {}
        ctx_keys = ["lat", "lon", "altitude_m", "heading", "ground_speed",
                    "distance_to_tower", "gimbal_pitch", "gimbal_roll",
                    "wind_speed", "air_temp", "humidity", "battery_remaining_pct",
                    "ir_min", "ir_max", "ir_avg"]
        return {k: frame.get(k) for k in ctx_keys if k in frame}

    def anomaly_frames(self):
        """Return sub-dataframe of frames where an anomaly was detected."""
        if "detected" not in self.df.columns:
            return pd.DataFrame()
        return self.df[self.df["detected"] == True].copy()

    def summary(self):
        """Print a quick mission summary."""
        if self.df.empty:
            print("Empty telemetry.")
            return
        print(f"Mission     : {self.df['mission_id'].iloc[0]}")
        print(f"Total frames: {len(self.df)}")
        print(f"Duration    : {len(self.df)/10:.1f} s at 10 Hz")
        if "altitude_m" in self.df.columns:
            print(f"Alt range   : {self.df['altitude_m'].min():.1f} – {self.df['altitude_m'].max():.1f} m AGL")
        anom = self.anomaly_frames()
        print(f"Anomaly frames: {len(anom)} / {len(self.df)}")


# Quick test with the first train mission file (if available)
train_tel_dir = TELEMETRY_DIR / "train" / "telemetry"
tel_csvs = sorted(train_tel_dir.glob("*.csv")) if train_tel_dir.exists() else []

if tel_csvs:
    parser = TelemetryParser(tel_csvs[0])
    parser.summary()
    print("\nContext for frame 10:")
    print(parser.get_context(10))
else:
    print("ℹ️  No telemetry CSVs found — placing dummy example.")
    parser = TelemetryParser("nonexistent.csv")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7.2 — Feature engineering from telemetry
# Derives stability, wind-stress and visibility scores per frame
# ─────────────────────────────────────────────────────────────
def engineer_features(df):
    """
    Derive three telemetry-based confidence modifiers:

    1. stability_score  (0–1): how steady the drone is
       Higher = more stable camera → more reliable detections

       stability = 1 - clip((|roll| + |pitch|) / 30°, 0, 1)

    2. wind_penalty (0–1): fraction of max tolerable wind
       wind_penalty = clip(wind_speed / 15 m/s, 0, 1)

    3. visibility_score (0–1): combined optical clarity proxy
       Requires: humidity < 90 %, distance_to_tower < 20 m
    """
    df = df.copy()

    # Stability
    if "roll" in df.columns and "pitch" in df.columns:
        df["stability_score"] = 1.0 - np.clip(
            (df["roll"].abs() + df["pitch"].abs()) / 30.0, 0, 1)
    else:
        df["stability_score"] = 1.0

    # Wind penalty
    if "wind_speed" in df.columns:
        df["wind_penalty"] = np.clip(df["wind_speed"] / 15.0, 0, 1)
    else:
        df["wind_penalty"] = 0.0

    # Visibility (distance + humidity)
    vis = pd.Series(np.ones(len(df)), index=df.index)
    if "distance_to_tower" in df.columns:
        vis *= np.clip(1 - (df["distance_to_tower"] - 5) / 20, 0.4, 1.0)
    if "humidity" in df.columns:
        vis *= np.where(df["humidity"] > 85, 0.8, 1.0)
    df["visibility_score"] = vis

    # Combined detection reliability modifier
    df["reliability"] = (df["stability_score"] * (1 - df["wind_penalty"]) * df["visibility_score"])

    return df


if not tel_train.empty:
    tel_train_fe = engineer_features(tel_train)
    print("Feature-engineered columns added:")
    print(tel_train_fe[["stability_score", "wind_penalty", "visibility_score", "reliability"]].describe().round(3))
else:
    print("ℹ️  No telemetry — skipping feature engineering.")

---

## 8. Multi-Modal Fusion Logic

### Fusion Strategy

We use **Late Fusion with Weighted Confidence Aggregation**. Each modality contributes a set of detection tuples:

$$\mathcal{D}_{RGB} = \{(c_i, s_i^{rgb}, b_i) \mid i = 1 \ldots N_{rgb}\}$$
$$\mathcal{D}_{thermal} = \{(c_j, s_j^{th}, b_j) \mid j = 1 \ldots N_{th}\}$$
$$\mathcal{D}_{tel} = \{(c_k, s_k^{tel}) \mid k = 1 \ldots N_{tel}\}$$

where $c$ is the class label, $s$ is the detection confidence, and $b$ is the bounding box.

**Fused Severity Score per anomaly:**

$$S_{fused} = w_{rgb} \cdot s^{rgb} \cdot \sigma_{rgb} + w_{th} \cdot s^{th} \cdot \sigma_{th} + w_{tel} \cdot s^{tel}$$

where $\sigma_{rgb}$ and $\sigma_{th}$ are the telemetry reliability modifiers from Section 7.2, and the weights are:

| Weight | Value | Rationale |
|---|---|---|
| $w_{rgb}$ | 0.45 | Primary visual defect signal |
| $w_{th}$ | 0.40 | Thermal complements RGB; catches hidden faults |
| $w_{tel}$ | 0.15 | Telemetry anomaly labels are secondary confirmation |

**IR Temperature Delta Adjustment:**

For detections where a thermal hotspot overlaps spatially with an RGB defect, we apply a boost:

$$S_{boosted} = S_{fused} \cdot \left(1 + \frac{\Delta T_{ir}}{50}\right)$$

where $\Delta T_{ir}$ is the IR temperature delta in Kelvin. A 15 K hotspot over a crack detection boosts the fused score by 30 %.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8.1 — Fusion engine
# ─────────────────────────────────────────────────────────────
W_RGB = 0.45
W_TH  = 0.40
W_TEL = 0.15

def fuse_detections(rgb_dets, thermal_dets, tel_context, reliability=1.0):
    """
    Fuse detections from all three modalities into a unified anomaly list.

    Parameters
    ----------
    rgb_dets     : list of dicts {class_name, confidence, bbox}
    thermal_dets : list of dicts {class_name, confidence, bbox, ir_temp_delta}
    tel_context  : dict with telemetry fields (lat, lon, altitude_m, …)
    reliability  : float [0,1] from engineer_features()

    Returns
    -------
    list of fused anomaly dicts, sorted by fused_score descending
    """
    fused = []

    # ── RGB detections ─────────────────────────────────────
    for det in rgb_dets:
        sev   = SEVERITY_MAP.get(det["class_name"], 0)
        score = W_RGB * det["confidence"] * reliability * sev / 4
        fused.append({
            "source"      : "RGB",
            "class"       : det["class_name"],
            "confidence"  : round(det["confidence"], 4),
            "severity_int": sev,
            "severity"    : SEVERITY_LABEL[sev],
            "bbox"        : det.get("bbox"),
            "fused_score" : round(score, 4),
            "ir_boost"    : 0.0,
            "lat"         : tel_context.get("lat"),
            "lon"         : tel_context.get("lon"),
            "altitude_m"  : tel_context.get("altitude_m"),
        })

    # ── Thermal detections ──────────────────────────────────
    for det in thermal_dets:
        sev      = SEVERITY_MAP.get(det["class_name"], 0)
        ir_delta = det.get("ir_temp_delta", 0) or 0
        boost    = 1 + ir_delta / 50.0
        score    = W_TH * det["confidence"] * reliability * sev / 4 * boost
        fused.append({
            "source"      : "Thermal",
            "class"       : det["class_name"],
            "confidence"  : round(det["confidence"], 4),
            "severity_int": sev,
            "severity"    : SEVERITY_LABEL[sev],
            "bbox"        : det.get("bbox"),
            "fused_score" : round(score, 4),
            "ir_boost"    : round(boost, 3),
            "lat"         : tel_context.get("lat"),
            "lon"         : tel_context.get("lon"),
            "altitude_m"  : tel_context.get("altitude_m"),
        })

    # ── Telemetry-only anomalies (no image bbox) ────────────
    if tel_context.get("detected"):
        cls  = tel_context.get("class", "no_anomaly")
        conf = float(tel_context.get("confidence", 0))
        sev  = SEVERITY_MAP.get(cls, 0)
        score= W_TEL * conf * sev / 4
        fused.append({
            "source"      : "Telemetry",
            "class"       : cls,
            "confidence"  : round(conf, 4),
            "severity_int": sev,
            "severity"    : SEVERITY_LABEL[sev],
            "bbox"        : None,
            "fused_score" : round(score, 4),
            "ir_boost"    : 0.0,
            "lat"         : tel_context.get("lat"),
            "lon"         : tel_context.get("lon"),
            "altitude_m"  : tel_context.get("altitude_m"),
        })

    # Sort by severity then score
    fused.sort(key=lambda x: (x["severity_int"], x["fused_score"]), reverse=True)
    return fused


print("✅ Fusion engine defined.")
print(f"   Weights: RGB={W_RGB}, Thermal={W_TH}, Telemetry={W_TEL}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8.2 — Visualise the fusion weight diagram
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Multi-Modal Fusion — Weight Distribution", fontsize=13, fontweight="bold")

# Pie chart
labels  = ["RGB Visual", "Thermal IR", "Telemetry"]
sizes   = [W_RGB, W_TH, W_TEL]
colors  = ["#457b9d", "#e63946", "#2a9d8f"]
explode = (0.05, 0.05, 0.05)
axes[0].pie(sizes, labels=labels, colors=colors, autopct="%1.0f%%",
            explode=explode, startangle=140, textprops={"fontsize": 11})
axes[0].set_title("Modality Contribution Weights", fontweight="bold")

# Score simulation: how different confidence + reliability combos affect fused score
reliabilities  = np.linspace(0.4, 1.0, 50)
conf_high      = 0.92
conf_medium    = 0.65
conf_low       = 0.40
sev_factor     = 1.0  # normalised severity = 1

axes[1].plot(reliabilities, W_RGB * conf_high   * reliabilities * sev_factor,
             color="#457b9d", lw=2.5, label=f"RGB conf={conf_high}")
axes[1].plot(reliabilities, W_RGB * conf_medium * reliabilities * sev_factor,
             color="#457b9d", lw=2, ls="--", label=f"RGB conf={conf_medium}")
axes[1].plot(reliabilities, W_TH  * conf_high   * reliabilities * sev_factor * 1.3,
             color="#e63946", lw=2.5, label=f"Thermal conf={conf_high} + IR boost")
axes[1].plot(reliabilities, W_TH  * conf_medium * reliabilities * sev_factor,
             color="#e63946", lw=2, ls="--", label=f"Thermal conf={conf_medium}")
axes[1].set_xlabel("Telemetry Reliability Score")
axes[1].set_ylabel("Fused Score Contribution")
axes[1].set_title("Score Contribution vs Reliability", fontweight="bold")
axes[1].legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig("fusion_weights.png", bbox_inches="tight")
plt.show()
print("Saved → fusion_weights.png")

---

## 9. Inference on Static Images

The inference pipeline takes a pair of matched RGB + thermal images (or just one if only one modality is available) plus an optional telemetry context dict, runs both YOLO models, fuses the results, and produces an annotated output image.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9.1 — Load trained models
# ─────────────────────────────────────────────────────────────
rgb_best_path     = Path("runs/rgb_tower/weights/best.pt")
thermal_best_path = Path("runs/thermal_tower/weights/best.pt")

rgb_infer_model     = YOLO(str(rgb_best_path))     if rgb_best_path.exists()     else None
thermal_infer_model = YOLO(str(thermal_best_path)) if thermal_best_path.exists() else None

if rgb_infer_model:
    print(f"✅ RGB model loaded     : {rgb_best_path}")
else:
    print("⚠️  RGB model not found — train first (Cell 5.2)")

if thermal_infer_model:
    print(f"✅ Thermal model loaded : {thermal_best_path}")
else:
    print("⚠️  Thermal model not found — train first (Cell 6.2)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9.2 — Single-image inference helper
# ─────────────────────────────────────────────────────────────
def run_inference(image_path, model, class_names, conf_threshold=0.30):
    """
    Run YOLO inference on a single image.

    Returns
    -------
    annotated_img : np.ndarray (RGB)
    detections    : list of dicts
    """
    results     = model(str(image_path), conf=conf_threshold, verbose=False)
    result      = results[0]
    annotated   = result.plot()                         # BGR annotated frame
    annotated   = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    detections = []
    for box in result.boxes:
        cls_id = int(box.cls.item())
        detections.append({
            "class_name" : class_names[cls_id] if cls_id < len(class_names) else str(cls_id),
            "confidence" : round(float(box.conf.item()), 4),
            "bbox"       : [round(v, 3) for v in box.xywhn.cpu().numpy().flatten().tolist()],
        })

    return annotated, detections

print("✅ Inference helper defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9.3 — Run inference on RGB test images
# ─────────────────────────────────────────────────────────────
rgb_test_dir = ROOT / "tower_inspection_dataset" / "test" / "images"
rgb_test_imgs = sorted(rgb_test_dir.glob("*.png"))[:4] if rgb_test_dir.exists() else []

if rgb_infer_model and rgb_test_imgs:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("RGB Model — Inference on Test Images", fontsize=13, fontweight="bold")

    all_rgb_dets = []
    for ax, img_path in zip(axes.flat, rgb_test_imgs):
        ann_img, dets = run_inference(img_path, rgb_infer_model, RGB_CLASSES)
        all_rgb_dets.append(dets)
        ax.imshow(ann_img)
        title_lines = [img_path.name]
        for d in dets[:3]:
            title_lines.append(f"  {d['class_name']} ({d['confidence']:.2f})")
        ax.set_title("\n".join(title_lines), fontsize=8, loc="left")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("rgb_inference_results.png", bbox_inches="tight")
    plt.show()
    print("Saved → rgb_inference_results.png")
    print(f"Total detections across 4 images: {sum(len(d) for d in all_rgb_dets)}")
else:
    print("ℹ️  RGB inference: model or test images not available — run after training.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9.4 — Run inference on Thermal test images
# ─────────────────────────────────────────────────────────────
th_test_dir  = ROOT / "cell_tower_thermal" / "test" / "images"
th_test_imgs = sorted(th_test_dir.glob("*.jpg"))[:4] if th_test_dir.exists() else []

if thermal_infer_model and th_test_imgs:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Thermal Model — Inference on Test Images", fontsize=13, fontweight="bold")

    all_th_dets = []
    for ax, img_path in zip(axes.flat, th_test_imgs):
        ann_img, dets = run_inference(img_path, thermal_infer_model, THERMAL_CLASSES)
        all_th_dets.append(dets)
        ax.imshow(ann_img)
        title_lines = [img_path.name]
        for d in dets[:3]:
            title_lines.append(f"  {d['class_name']} ({d['confidence']:.2f})")
        ax.set_title("\n".join(title_lines), fontsize=8, loc="left")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("thermal_inference_results.png", bbox_inches="tight")
    plt.show()
    print("Saved → thermal_inference_results.png")
else:
    print("ℹ️  Thermal inference: model or test images not available — run after training.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 9.5 — Side-by-side fused output (RGB + Thermal)
# ─────────────────────────────────────────────────────────────
def show_fused_pair(rgb_img_path, thermal_img_path,
                    tel_context=None, reliability=1.0,
                    save_path="fused_output.png"):
    """
    Run both models on a matched RGB+thermal pair, fuse results,
    and display side-by-side with a printed anomaly table.
    """
    tel_context = tel_context or {}

    # Inference
    rgb_ann, rgb_dets, th_ann, th_dets = None, [], None, []
    if rgb_infer_model and Path(rgb_img_path).exists():
        rgb_ann, rgb_dets  = run_inference(rgb_img_path,     rgb_infer_model,     RGB_CLASSES)
    if thermal_infer_model and Path(thermal_img_path).exists():
        th_ann,  th_dets   = run_inference(thermal_img_path, thermal_infer_model, THERMAL_CLASSES)

    # Fusion
    fused = fuse_detections(rgb_dets, th_dets, tel_context, reliability)

    # Plot
    fig = plt.figure(figsize=(16, 7))
    gs  = gridspec.GridSpec(1, 3, width_ratios=[5, 5, 4], figure=fig)
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1])
    ax2 = fig.add_subplot(gs[2])

    if rgb_ann is not None:
        ax0.imshow(rgb_ann)
        ax0.set_title(f"RGB — {Path(rgb_img_path).name}", fontweight="bold", fontsize=9)
    else:
        ax0.text(0.5, 0.5, "RGB not available", ha="center", va="center", transform=ax0.transAxes)
    ax0.axis("off")

    if th_ann is not None:
        ax1.imshow(th_ann)
        ax1.set_title(f"Thermal — {Path(thermal_img_path).name}", fontweight="bold", fontsize=9)
    else:
        ax1.text(0.5, 0.5, "Thermal not available", ha="center", va="center", transform=ax1.transAxes)
    ax1.axis("off")

    # Anomaly table
    ax2.axis("off")
    table_data  = [["#", "Source", "Class", "Conf", "Severity", "Score"]]
    for i, f in enumerate(fused[:8], 1):
        table_data.append([
            str(i),
            f["source"],
            f["class"],
            f"{f['confidence']:.2f}",
            f["severity"],
            f"{f['fused_score']:.3f}",
        ])
    tbl = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                    loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.5)
    ax2.set_title("Fused Anomaly Report", fontweight="bold", fontsize=10)

    fig.suptitle("Multi-Modal Fusion Output", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")
    return fused


# Run on first matched test pair
if rgb_test_imgs and th_test_imgs:
    dummy_tel = {"lat": 22.5726, "lon": 88.3639, "altitude_m": 45.0,
                 "detected": False, "ir_temp_delta": 12.0}
    fused_output = show_fused_pair(
        rgb_test_imgs[0], th_test_imgs[0],
        tel_context=dummy_tel, reliability=0.92,
        save_path="fused_output.png"
    )
else:
    print("ℹ️  Test images not available — run after placing the datasets.")
    fused_output = []

---

## 10. Severity Scoring & Risk Matrix

Each fused anomaly is placed on a 2D **Risk Matrix** using:

- **X-axis** → Detection confidence (probability of true anomaly)
- **Y-axis** → Intrinsic severity (1–4 scale from class metadata)

The quadrants define the inspection priority:

| Quadrant | Confidence | Severity | Action |
|---|---|---|---|
| I — CRITICAL | High | High | Immediate ground inspection |
| II — MONITOR | Low | High | Re-fly for confirmation |
| III — LOG | High | Low | Log for next maintenance cycle |
| IV — DISCARD | Low | Low | Likely false positive |

### Final Priority Index

$$\text{PI} = \frac{\text{severity} \times \text{confidence} \times S_{fused}}{4}$$

Anomalies with PI > 0.7 trigger an immediate alert.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 10.1 — Risk matrix visualisation
# ─────────────────────────────────────────────────────────────
def plot_risk_matrix(fused_list, title="Risk Matrix", save_path=None):
    if not fused_list:
        print("No fused detections to plot.")
        return

    fig, ax = plt.subplots(figsize=(9, 7))

    # Background quadrant shading
    ax.axhspan(2.5, 4.5, xmin=0.5, xmax=1.0, alpha=0.12, color="red",    label="_nolegend_")
    ax.axhspan(2.5, 4.5, xmin=0.0, xmax=0.5, alpha=0.08, color="orange", label="_nolegend_")
    ax.axhspan(0.5, 2.5, xmin=0.5, xmax=1.0, alpha=0.08, color="blue",   label="_nolegend_")
    ax.axhspan(0.5, 2.5, xmin=0.0, xmax=0.5, alpha=0.05, color="green",  label="_nolegend_")

    # Quadrant labels
    ax.text(0.75, 3.9, "I — CRITICAL",    ha="center", color="#b22222", fontsize=9, fontweight="bold")
    ax.text(0.25, 3.9, "II — MONITOR",   ha="center", color="#d4612a", fontsize=9, fontweight="bold")
    ax.text(0.75, 1.1, "III — LOG",       ha="center", color="#1a4fa0", fontsize=9, fontweight="bold")
    ax.text(0.25, 1.1, "IV — DISCARD",   ha="center", color="#2d6a2d", fontsize=9, fontweight="bold")

    color_map = {"RGB": "#457b9d", "Thermal": "#e63946", "Telemetry": "#2a9d8f"}
    marker_map= {"RGB": "o",       "Thermal": "s",        "Telemetry": "^"}  

    for f in fused_list:
        c = color_map.get(f["source"], "gray")
        m = marker_map.get(f["source"], "o")
        s = f["fused_score"] * 600 + 60
        ax.scatter(f["confidence"], f["severity_int"],
                   c=c, marker=m, s=s, alpha=0.85, edgecolors="white", linewidths=0.8)
        ax.annotate(f["class"], (f["confidence"], f["severity_int"]),
                    textcoords="offset points", xytext=(5, 3), fontsize=7)

    # Legend
    handles = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
    ax.legend(handles=handles, title="Source", loc="lower right", fontsize=9)

    ax.set_xlim(0, 1)
    ax.set_ylim(0.5, 4.5)
    ax.set_xlabel("Detection Confidence", fontweight="bold")
    ax.set_ylabel("Severity Level (1=Low … 4=URGENT)", fontweight="bold")
    ax.set_yticks([1, 2, 3, 4])
    ax.set_yticklabels(["🟢 Low", "🟡 Medium", "🟠 High", "🔴 URGENT"])
    ax.axvline(0.5, color="gray", ls="--", alpha=0.5)
    ax.axhline(2.5, color="gray", ls="--", alpha=0.5)
    ax.set_title(title, fontsize=13, fontweight="bold")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


if fused_output:
    plot_risk_matrix(fused_output, title="Risk Matrix — Test Frame", save_path="risk_matrix.png")
else:
    # Demo with synthetic detections if no real ones yet
    demo = [
        {"source": "RGB",      "class": "crack",            "confidence": 0.91, "severity_int": 4, "severity": "🔴", "fused_score": 0.38},
        {"source": "Thermal",  "class": "hotspot_critical", "confidence": 0.88, "severity_int": 4, "severity": "🔴", "fused_score": 0.35},
        {"source": "RGB",      "class": "corrosion",        "confidence": 0.74, "severity_int": 3, "severity": "🟠", "fused_score": 0.25},
        {"source": "Thermal",  "class": "hotspot_moderate", "confidence": 0.65, "severity_int": 3, "severity": "🟠", "fused_score": 0.22},
        {"source": "RGB",      "class": "rust",             "confidence": 0.82, "severity_int": 2, "severity": "🟡", "fused_score": 0.14},
        {"source": "RGB",      "class": "paint_damage",     "confidence": 0.55, "severity_int": 1, "severity": "🟢", "fused_score": 0.06},
        {"source": "Telemetry","class": "antenna_misalign.","confidence": 0.40, "severity_int": 3, "severity": "🟠", "fused_score": 0.05},
    ]
    plot_risk_matrix(demo, title="Risk Matrix — Demo (Synthetic)", save_path="risk_matrix.png")

---

## 11. Georeferenced Inspection Report

The final output of each inspection run is a georeferenced JSON report listing all detected anomalies with GPS coordinates, plus an interactive Folium map with colour-coded severity markers.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11.1 — Build and save JSON inspection report
# ─────────────────────────────────────────────────────────────
def build_report(fused_list, mission_id="M001", site_id="Kolkata_Tower_01"):
    urgents  = [f for f in fused_list if f["severity_int"] >= 4]
    highs    = [f for f in fused_list if f["severity_int"] == 3]
    mediums  = [f for f in fused_list if f["severity_int"] == 2]

    report = {
        "mission_id"    : mission_id,
        "site_id"       : site_id,
        "generated_at"  : pd.Timestamp.now().isoformat(),
        "summary": {
            "total_anomalies" : len(fused_list),
            "urgent"          : len(urgents),
            "high"            : len(highs),
            "medium"          : len(mediums),
        },
        "anomalies": fused_list
    }

    with open("inspection_report.json", "w") as f:
        json.dump(report, f, indent=2, default=str)

    print("📋 Inspection Report")
    print(f"   Mission     : {mission_id}")
    print(f"   Site        : {site_id}")
    print(f"   Anomalies   : {len(fused_list)}")
    print(f"   🔴 URGENT   : {len(urgents)}")
    print(f"   🟠 High     : {len(highs)}")
    print(f"   🟡 Medium   : {len(mediums)}")
    print("   Saved → inspection_report.json")
    return report


active_fused = fused_output if fused_output else demo
report = build_report(active_fused)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11.2 — Generate Folium geo-map
# ─────────────────────────────────────────────────────────────
def generate_geo_map(fused_list, center_lat=22.5726, center_lon=88.3639, save_path="inspection_map.html"):
    m = folium.Map(location=[center_lat, center_lon], zoom_start=17,
                   tiles="CartoDB positron")

    SEVERITY_COLORS = {4: "red", 3: "orange", 2: "beige", 1: "green", 0: "blue"}
    SEVERITY_ICONS  = {4: "warning-sign", 3: "info-sign", 2: "ok-circle", 1: "ok", 0: "ok-circle"}

    for i, f in enumerate(fused_list):
        lat = f.get("lat") or center_lat + np.random.uniform(-0.0001, 0.0001)
        lon = f.get("lon") or center_lon + np.random.uniform(-0.0001, 0.0001)
        sev = f["severity_int"]
        popup_html = f"""
        <b>#{i+1} {f['class']}</b><br>
        Source     : {f['source']}<br>
        Confidence : {f['confidence']:.2f}<br>
        Severity   : {f['severity']}<br>
        Fused Score: {f['fused_score']:.3f}<br>
        Alt (m)    : {f.get('altitude_m', 'N/A')}
        """
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(popup_html, max_width=220),
            tooltip=f['class'],
            icon=folium.Icon(color=SEVERITY_COLORS[sev],
                             icon=SEVERITY_ICONS[sev], prefix="glyphicon")
        ).add_to(m)

    m.save(save_path)
    print(f"✅ Interactive map saved → {save_path}")
    print("   Open the .html file in a browser to explore georeferenced anomalies.")
    return m


geo_map = generate_geo_map(active_fused)
display(geo_map)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 11.3 — Summary dashboard — all metrics in one chart
# ─────────────────────────────────────────────────────────────
sev_counts = {"🔴 URGENT": 0, "🟠 High": 0, "🟡 Medium": 0, "🟢 Low": 0, "✅ Normal": 0}
src_counts = {"RGB": 0, "Thermal": 0, "Telemetry": 0}
for f in active_fused:
    sev_counts[f["severity"]] = sev_counts.get(f["severity"], 0) + 1
    src_counts[f["source"]]   = src_counts.get(f["source"], 0) + 1

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Inspection Session Summary", fontsize=14, fontweight="bold")

# Severity breakdown
sev_labels  = [k for k,v in sev_counts.items() if v > 0]
sev_vals    = [v for v in sev_counts.values() if v > 0]
sev_colors  = ["#e63946", "#f4a261", "#e9c46a", "#2a9d8f", "#a8dadc"]
axes[0].bar(range(len(sev_labels)), sev_vals,
            color=sev_colors[:len(sev_labels)], edgecolor="white")
axes[0].set_xticks(range(len(sev_labels)))
axes[0].set_xticklabels(sev_labels, rotation=20, fontsize=9)
axes[0].set_title("Anomalies by Severity", fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(sev_vals):
    axes[0].text(i, v + 0.05, str(v), ha="center", fontweight="bold")

# Source breakdown pie
src_labels = [k for k,v in src_counts.items() if v > 0]
src_vals   = [v for k,v in src_counts.items() if v > 0]
axes[1].pie(src_vals, labels=src_labels, autopct="%1.0f%%",
            colors=["#457b9d", "#e63946", "#2a9d8f"],
            startangle=120, textprops={"fontsize": 10})
axes[1].set_title("Detections by Source", fontweight="bold")

# Fused score distribution
scores = [f["fused_score"] for f in active_fused]
axes[2].hist(scores, bins=10, color="#264653", edgecolor="white", alpha=0.85)
axes[2].axvline(np.mean(scores), color="#e63946", ls="--", lw=2, label=f"Mean={np.mean(scores):.3f}")
axes[2].set_title("Fused Score Distribution", fontweight="bold")
axes[2].set_xlabel("Fused Score")
axes[2].set_ylabel("Count")
axes[2].legend()

plt.tight_layout()
plt.savefig("summary_dashboard.png", bbox_inches="tight")
plt.show()
print("Saved → summary_dashboard.png")

---

## 12. Key Formulas & Theoretical Grounding

This section consolidates all mathematical foundations in one place for reference.

---

### 12.1 Object Detection Metrics

**Precision and Recall**

$$P = \frac{TP}{TP + FP} \qquad R = \frac{TP}{TP + FN}$$

- $TP$ = predicted box matched to a ground-truth box with IoU ≥ threshold
- $FP$ = predicted box that matched nothing
- $FN$ = ground-truth box that no prediction covered

**F1 Score**

$$F_1 = 2 \cdot \frac{P \cdot R}{P + R}$$

**Average Precision (AP)**

$$AP = \sum_{k=1}^{N} P(k) \cdot \Delta R(k)$$

Computed as the area under the interpolated precision–recall curve. YOLOv8 uses the 101-point interpolation scheme.

---

### 12.2 Telemetry Reliability

$$\text{stability}(t) = 1 - \text{clip}\!\left(\frac{|\phi(t)| + |\theta(t)|}{30°}, 0, 1\right)$$

where $\phi$ = roll and $\theta$ = pitch in degrees. A hovering drone at 0° roll / 0° pitch gives stability = 1.0.

$$\text{reliability}(t) = \text{stability}(t) \cdot (1 - \text{wind\_penalty}(t)) \cdot \text{visibility}(t)$$

---

### 12.3 Multi-Modal Fusion Score

$$S_{fused} = \underbrace{0.45 \cdot s_{rgb} \cdot \sigma}_{\text{visual}} + \underbrace{0.40 \cdot s_{th} \cdot \sigma \cdot \left(1 + \frac{\Delta T}{50}\right)}_{\text{thermal + IR boost}} + \underbrace{0.15 \cdot s_{tel}}_{\text{telemetry}}$$

where $\sigma$ = reliability score, $\Delta T$ = IR temperature delta in Kelvin.

---

### 12.4 Priority Index

$$\text{PI} = \frac{\text{sev}(c) \times s \times S_{fused}}{4}$$

- If PI > 0.70 → **immediate ground inspection required**
- If 0.40 < PI ≤ 0.70 → **schedule maintenance within 7 days**
- If PI ≤ 0.40 → **log and monitor next cycle**

---

### 12.5 Cosine Annealing LR Schedule

$$\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})\left(1 + \cos\frac{t}{T}\pi\right)$$

where $t$ is the current epoch, $T$ is the total epochs, $\eta_{max}$ = `lr0`, $\eta_{min}$ = `lr0 * lrf`. The 3-epoch warm-up linearly ramps from $10^{-4}$ to $\eta_{max}$ before the cosine decay begins.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 12.1 — Visualise LR schedule & IoU loss curve
# ─────────────────────────────────────────────────────────────
epochs = np.arange(0, 100)
lr0 = 1e-3; lrf = 0.01; warmup = 3

lrs = []
for e in epochs:
    if e < warmup:
        lrs.append(lr0 * (e + 1) / warmup)
    else:
        t_frac = (e - warmup) / (len(epochs) - warmup)
        lrs.append(lr0 * lrf + 0.5 * (lr0 - lr0 * lrf) * (1 + np.cos(np.pi * t_frac)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Schedule & Loss Dynamics", fontsize=13, fontweight="bold")

axes[0].plot(epochs, lrs, color="#457b9d", lw=2.5)
axes[0].axvspan(0, warmup, alpha=0.15, color="#f4a261", label="Warm-up phase")
axes[0].set_title("Cosine Annealing LR Schedule", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Learning Rate")
axes[0].legend()

# Simulated CIoU loss decay
np.random.seed(42)
box_loss = 2.5 * np.exp(-0.04 * epochs) + 0.1 + np.random.normal(0, 0.03, len(epochs))
cls_loss = 1.8 * np.exp(-0.035* epochs) + 0.08+ np.random.normal(0, 0.025, len(epochs))
axes[1].plot(epochs, box_loss, color="#e63946", lw=2, label="Box (CIoU) Loss")
axes[1].plot(epochs, cls_loss, color="#2a9d8f", lw=2, label="Class (BCE) Loss")
axes[1].set_title("Loss Convergence (Illustrative)", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("lr_and_loss.png", bbox_inches="tight")
plt.show()
print("Saved → lr_and_loss.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 12.2 — Priority Index computation & table
# ─────────────────────────────────────────────────────────────
def compute_priority_index(fused_list):
    rows = []
    for f in fused_list:
        pi  = (f["severity_int"] * f["confidence"] * f["fused_score"]) / 4
        if pi > 0.70:
            action = "🚨 Immediate inspection"
        elif pi > 0.40:
            action = "📅 Schedule within 7 days"
        else:
            action = "📝 Log for next cycle"
        rows.append({
            "Class"      : f["class"],
            "Source"     : f["source"],
            "Confidence" : f["confidence"],
            "Severity"   : f["severity"],
            "Fused Score": round(f["fused_score"], 4),
            "PI"         : round(pi, 4),
            "Action"     : action,
        })
    df = pd.DataFrame(rows).sort_values("PI", ascending=False).reset_index(drop=True)
    df.index += 1
    return df


pi_df = compute_priority_index(active_fused)
print("\nAnomaly Priority Index Table")
print("=" * 70)
display(pi_df.style
        .background_gradient(subset=["PI"], cmap="RdYlGn_r")
        .set_properties(**{"text-align": "center"}))

---

## Appendix A — Replay Video Section (Commented)

The code block below processes a video file frame-by-frame through both YOLO models and writes an annotated output video. Uncomment and point to your `.mp4` or `.avi` inspection footage to use it.

In [ ]:
# ─────────────────────────────────────────────────────────────
# APPENDIX A — Video replay (uncomment to use)
# ─────────────────────────────────────────────────────────────

# VIDEO_INPUT_PATH  = "path/to/your/inspection_video.mp4"
# VIDEO_OUTPUT_PATH = "annotated_output.mp4"

# cap = cv2.VideoCapture(VIDEO_INPUT_PATH)
# fps = cap.get(cv2.CAP_PROP_FPS)
# W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# out = cv2.VideoWriter(VIDEO_OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

# frame_id = 0
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     # RGB inference
#     if rgb_infer_model:
#         rgb_res = rgb_infer_model(frame, conf=0.30, verbose=False)[0]
#         frame   = rgb_res.plot()

#     # Thermal: would require a paired thermal frame
#     # If running on a single-camera RGB video, skip thermal

#     # Overlay frame counter
#     cv2.putText(frame, f"Frame {frame_id:05d}", (20, 40),
#                 cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)

#     out.write(frame)
#     frame_id += 1

# cap.release()
# out.release()
# print(f"✅ Video written to: {VIDEO_OUTPUT_PATH}")

print("Replay video section is commented out.")
print("Uncomment and set VIDEO_INPUT_PATH to process a video file.")

---

## Appendix B — Output Files Reference

| File | Contents |
|---|---|
| `class_distribution.png` | Bar charts of annotation counts per class |
| `rgb_samples.png` | Grid of RGB ground-truth annotated images |
| `thermal_samples.png` | Grid of thermal annotated images |
| `telemetry_signals.png` | Altitude / battery / speed traces for one mission |
| `rgb_training_curves.png` | Loss and mAP over 100 training epochs (RGB model) |
| `thermal_training_curves.png` | Loss and mAP over 100 epochs (Thermal model) |
| `fusion_weights.png` | Weight pie chart + reliability vs score plot |
| `rgb_inference_results.png` | Annotated test-set predictions (RGB model) |
| `thermal_inference_results.png` | Annotated test-set predictions (Thermal model) |
| `fused_output.png` | Side-by-side RGB + thermal with fused anomaly table |
| `risk_matrix.png` | 2D risk scatter plot |
| `summary_dashboard.png` | 3-panel inspection summary |
| `lr_and_loss.png` | LR schedule + loss convergence illustration |
| `inspection_report.json` | Machine-readable anomaly report |
| `inspection_map.html` | Interactive Folium georeferenced map |
| `runs/rgb_tower/weights/best.pt` | Best RGB model checkpoint |
| `runs/thermal_tower/weights/best.pt` | Best Thermal model checkpoint |

---

## Appendix C — Extending the Pipeline

**Swapping the backbone** — replace `yolov8n.pt` with `yolov8s.pt` or `yolov8m.pt` for higher accuracy at the cost of inference speed. On an NVIDIA T4, the nano model runs at ~90 FPS; the small model ~55 FPS; medium ~30 FPS.

**Adding a depth channel** — if a stereo or LiDAR sensor is available, depth maps can be stacked as a 4th channel and a custom YOLOv8 head can be fine-tuned to incorporate 3D information for sub-centimetre crack localisation.

**Real-time edge deployment** — export either model to ONNX or TensorRT with `model.export(format='engine')` for deployment on Jetson Orin or similar edge compute modules mounted on the drone.

**Active learning loop** — uncertain predictions (confidence 0.25–0.45) can be streamed to a labelling queue, enabling the model to improve on hard cases encountered in real inspections.